In [ ]:
from collections import Counter
import gzip
import pathlib

import pandas as pd
import scipy
import numpy as np

import scanpy as sc
import anndata as ad
import gseapy

import seaborn as sns
import matplotlib.pyplot as plt

import flipcrow
import flipcrow.paths
import flipcrow.scp1644_lib

# sc.settings.verbosity = 0
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
# #sc._settings.ScanpyConfig.n_jobs = -1
# print(sc._settings.ScanpyConfig.n_jobs)
# # dir(sc._settings.ScanpyConfig.n_jobs)

In [ ]:
# from scanpy import _version
# # help(sc._version)
# print(sc._version.version)

In [ ]:
print(list((flipcrow.paths.DATA_PATH / 'SCP1644').glob('*')))

In [ ]:
# don't scale

outfile = 'GGB_scp1644_abl_trim_savepoint.h5ad.hdf5'
abl_trim = ad.read_h5ad(str(flipcrow.paths.DATA_PATH / "SCP1644" / outfile))

In [ ]:
dir(abl_trim.raw)

In [ ]:
abl_old = abl_trim.copy()

sc.pp.pca(abl_trim)
sc.pp.neighbors(abl_trim, n_neighbors=40, n_pcs=45, use_rep='X_pca')
sc.tl.leiden(abl_trim)

In [ ]:
sc.tl.rank_genes_groups(abl_trim, 'leiden', method='wilcoxon', tie_correct=True)


In [ ]:
sc.pl.rank_genes_groups(abl_trim, n_genes=30, sharey=False)

In [ ]:
# identify malignant cells

import itertools
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
pdac_subtypes = {
    'EXOCRINE_LIKE': ['REG1B', 'REG3A', 'REG1A', 'PNLIPRP2', 'CEL', 'PNLIP', 'PLA2G1B', 'CELA3A', 'CPB1', 'CELA3B', 'CTRB2', 'CLPS', 'CELA2B', 'PRSS2', 'PRSS1', 'GP2', 'SLC3A1', 'CFTR', 'SLC4A4', 'SPINK1'],
    'CLASSICAL': ['AIM2', 'FAM26F', 'GPM6B', 'S100A2', 'KRT14', 'CAV1', 'LOX', 'SLC2A3', 'TWIST1', 'PAPPA', 'NT5E', 'CKS2', 'HMMR', 'SLC5A3', 'PMAIP1', 'PHLDA1', 'SLC16A1', 'FERMT1', 'HK2', 'AHNAK2'],
    'QM_PDA': ['TMEM45B', 'SDR16C5', 'GPRC5A', 'AGR2', 'S100P', 'FXYD3', 'ST6GALNAC1', 'CEACAM5', 'CEACAM6', 'TFF1', 'TFF3', 'CAPN8', 'FOXQ1', 'ELF3', 'ERBB3', 'TSPAN8', 'TOX3', 'LGALS4', 'PLS1', 'GPX2', 'ATP10B', 'MUC13'],
}

# print(list(itertools.chain(*list(pdac_subtypes.values()))))
# bop = list(itertools.chain(*list(pdac_subtypes.values())))
all_markers = [x for x in list(itertools.chain(*list(pdac_subtypes.values()))) if x in abl_trim.var_names]
# all_markers_amt = abl_trim.X.loc[:, pdac_subtypes]
# print(all_markers_amt)

# total_all_markers = abl_trim.X[:, pdac_subtypes]

# sc.pl.tsne(abl_trim, color=all_markers)

for k in pdac_subtypes:
    sc.pl.tsne(abl_trim, color=[x for x in pdac_subtypes[k] if x in abl_trim.var_names])



In [ ]:
abl_trim.obs['pdac_marker_sum'] = abl_trim[:, all_markers].X.sum(axis=1)
abl_trim.obs['pdac_marker_max'] = abl_trim[:, all_markers].X.max(axis=1)

# mean and median not interesting

abl_trim.obs['pdac_marker_mean'] = abl_trim[:, all_markers].X.mean(axis=1)

# this test works very well as a signature. filters noise
abl_trim.obs['pdac_marker_score'] = abl_trim.obs['pdac_marker_max'] - abl_trim.obs['pdac_marker_sum']

In [ ]:
sc.pl.tsne(abl_trim, color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")
sc.pl.tsne(abl_trim, color=['leiden', 'donor_ID'])
sc.pl.tsne(abl_trim, color=['leiden'])

sc.pl.tsne(abl_trim, color=['pdac_marker_score', 'Coarse_Cell_Annotations'])
sc.pl.tsne(abl_trim, color=['pdac_marker_max', 'Coarse_Cell_Annotations'])
sc.pl.tsne(abl_trim, color=['pdac_marker_score', 'Coarse_Cell_Annotations'])

In [ ]:
coarse_marker_genes = {
    'GGB:MACROPHAGE_TREM2+_M2': [  # Group 0
        'SLCO2B1', # M2 polarization marker
        'CD163', # M2 polarization marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6525806/
        'GPNMB', # M2 polarization marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6525806/
        'TREM2', # TREM2+ immuno suppressive TAM marker https://www.nature.com/articles/s41590-023-01475-4
        'MSR1', # macrophage scavenger receptor
        'C1QC', # immuno suppressive TAM marker
        'C1QB', # immuno suppressive TAM marker
        'C1QA', # immuno suppressive TAM marker
        'MS4A4A', # M2 macrophages (not M1)
        'APOC1', # TAM marker
        'SIGLEC1', # CD169+ macrophage marker
    ],
    'GGB:CD8_Tem_Teff': [ # Group 1
        'CD8A', # CD8+
        'GZMA', # granzyme A (Tem, Teff) https://www.biocompare.com/Editorial-Articles/569888-A-Guide-to-T-Cell-Markers/
        'GZMK', # granzyme K
        'GZMH', # granzyme H
        'CD3D', # T cell
        'EOMES', # Tcm, Tem, Teff https://www.biocompare.com/Editorial-Articles/569888-A-Guide-to-T-Cell-Markers/
        'CD8B', # CD8+
        'LCK', # CD4+ CD8+
        'TRGC2',  # TCRVγ(non-9)Vδ1 cytotoxic T cell marker
        'IL2RB', # Tscm, Tcm, Tem, Teff	https://www.biocompare.com/Editorial-Articles/569888-A-Guide-to-T-Cell-Markers/
    ],
    'GGB:CD4_Tnaive_Treg': [ # Group 2, low FOXP3
        'IL7R', # Tcell
        'SPOCK2', # exhausted CD8?
        'TRAC', # T cell receptor
        'TRBC2', # T cell receptor
        'TNFRSF25', # Treg CD4+
        'CD40L', # activated T cells (CD4?)
        'RPS3', # Treg
        'CD5', # inhibitory receptor
        'ETS1', # Treg
        'CAMK4', # T cell activation
        'RPS3', # high ribosomes in naive CD4+ T lymphocytes
        'RPS12', # high ribosomes in naive CD4+ T lymphocytes
        'RPL3', # high ribosomes in naive CD4+ T lymphocytes
        'RPL11', # high ribosomes in naive CD4+ T lymphocytes
        'RPLP2', # high ribosomes in naive CD4+ T lymphocytes
    ],
    'GGB:MACROPHAGE_CD14': [ # Group 3 Macrophage
        'FCN1', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6728754/, 
        'CD300E', # https://pubmed.ncbi.nlm.nih.gov/20039296/
        'S100A8', # M2 polarization
        'S100A9', # M2 polarization
        'MNDA', # Oppose macrophage differentiation and activation https://pubmed.ncbi.nlm.nih.gov/7890814/
        'LILRB2', # M2 polarization Macrophage/MDSC
        'CLEC12A', # M2 polarization
    ],
    'GGB:CD4_Tnaive_Treg_2': [ # Group 4, similar to group 2, low FOXP3
        'BCL2', # Treg via https://molecular-cancer.biomedcentral.com/articles/10.1186/s12943-022-01516-w
        'IKZF1', # Treg cooperator with FOXP3
    ],
    'GGB:CAF': [ # Group 5, either cancer or CAF, probably CAF?
        'KRT81', # QM/squamous/basal-like https://pubmed.ncbi.nlm.nih.gov/29101303/
        'ANKRD1', # https://www.nature.com/articles/s41467-024-45308-w,
        'COL8A1', # CAF migration/invasion enhancer https://pubmed.ncbi.nlm.nih.gov/36375776/
        'SPOCK1', # CAF stroma
        'EFEMP1', # CAF stroma
        'APCDD1L', # aggressive cancer marker
        'TNC', # stromal fibroblast
        'MXRA8', # CAF marker https://pubmed.ncbi.nlm.nih.gov/35020975/
    ],
    'GGB:NK_Teff': [ # Group 6, T_NK cell (T or NK?)
        'CD160', # https://pubmed.ncbi.nlm.nih.gov/35020975/
        'TIGIT', # exhaustion marker
        'NKG7', # NK and T cell marker
        'TOX', # NK cell differntiation maturation
        'TOX2', # NK cell differentiation maturation
    ],
    'GGB:PDAC_CL1_BASAL': [ # Group 7, PDAC cluster (basal) 1
        'CLDN18', # PDAC marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9907975/
        'AGR3', # basal PDAC marker https://www.nature.com/articles/s41588-022-01157-1
        'BTNL8', # basal PDAC marker 
        'CDH17', # pdac marker
        'REG4', # pdac GATA6 target
        'GPX2', # promoter of EMT invasion metastasis expressing Wnt
        'SPINK1', # EGFR promoter in PDAC
        'CDHR2', # Ductal cells
    ],  
    'GGB:PDAC_CL2_BASAL': [ # Group 8, PDAC cluster (classical/basal ambiguous, no DDC) 2 
        'SPINK1', # EGFR promoter in PDAC
        'LCN2', # PDAC promoter inflammation regulator https://pubmed.ncbi.nlm.nih.gov/28249896/
        'UPK1B', # squamous Wnt/B-catenin https://pubmed.ncbi.nlm.nih.gov/28249896/
        'S100P', # only in tumor epithelium https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8435722/
        'S100A6', # EMT promoter
        'FXYD3', # PDAC growth promoter https://pubmed.ncbi.nlm.nih.gov/16003754/
        'TFF2', # enriched in PDAC https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7605628/ but classical marker
        'TM4SF1', # migration invasion promoter
        'AGR2', # classical marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7664990/
        'LGALS4', # classical marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7664990/
        'ANXA10', # classical marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7664990/
    ],
    'GGB:DC2_CD1C_CLEC10A_FCER1A': [ # Group 9 CD1C+ CLEC10A+ FCER1A+ DC2 dendritic cells
        'CD1C', # DC2 marker https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2020.559166/full
        'CLEC10A', # DC2 marker
        'FCER1A', # DC2 marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6445737/ https://www.nature.com/articles/s41698-023-00455-z
    ],
    'GGB:B_CELL_NAIVE_ACTIVATED': [ # Group 10 CD19+ IgD+ IgM+ CD27- CD38- CD24(CR2)- https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6813733/
        'FCRL1', # B cell https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6864034/
        'FCRLA', # B cell w/ high levels of IG plasma phenotype https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3250215/
        'CD79A', # B cell receptor complex
        'IGHD', # immunoglobulin https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6813733 see typing methods here 
        'CD22', # B cell inhibitory pathway, B cell only
        'CD19', # B cell core marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6813733
        'FCRL5', # Naive B cell activation marker
        'MS4A1', # B cell CD20
    ],
    'GGB:PDAC_CL3_TP63+_BASAL': [ # Group 11 TP63+ basal
        'KRT6A', # PDAC TAM effects https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7746340/
        'TP63', # PDAC squamous type https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6296757/
        'PKP1', # squamous cell lung cancer https://www.nature.com/articles/s41388-019-1129-3
        'KRT5', # squamous/basal https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7664990/
        'S100A2', # basal marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7664990/
    ],
    'GGB:NEUROENDOCRINE_TUMOR': [ # Group 12 neuroendocrine tumor
        'PCSK2', # neuroendocrine marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7702075/
        'SCG3', # neuroendocrine marker https://www.nature.com/articles/6604565
        'KCNH6', # neuroendocrine marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9950345/
        'CPLX2', # neuroendocrine marker https://pubmed.ncbi.nlm.nih.gov/23912489/
    ],
    'GGB:PDAC_CL4': [ # Group 13 PDAC (ENRICHR KRAS signaling up, secondary malignant neoplasm of pancreas, Pancreas cancer cell line encyclopedia)
        'CXCL17', # IPMA (PDAC precursor) marker https://pubmed.ncbi.nlm.nih.gov/20955708/
        'CXCL14', # upregulated in PDAC https://translational-medicine.biomedcentral.com/articles/10.1186/1479-5876-11-6
        'PADI1', # hypoxia sensitive stimulation of glycolysis in cancer https://www.nature.com/articles/s41467-021-21960-4
        'DPCR1', # upregulated or downregulated in tumor vs nontumor tissues, contradictory, https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6403508/ https://pubmed.ncbi.nlm.nih.gov/29242154/
        'PSCA', # found on 60% of pancreatic cancers https://ascopubs.org/doi/10.1200/JCO.2020.38.4_suppl.734
        'MUC16', # metastatic promoter in PDAC https://pubmed.ncbi.nlm.nih.gov/27382435/
        'LYPD2', # associated with poor survival in PDAC along with PSCA https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7869573/
    ],
    'GGB:PDAC_CL5_CLASSICAL': [ # Group 14 Classical PDAC
        'ANXA10', # classical PDAC marker
        'AGR2', # classical PDAC marker
        'CEACAM6', # PDAC marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8226832/
        'SPINK1', # PDAC promoter
        'LGALS4', # classical PDAC promoter
    ],
    'GGB:PC_PCreg_IgA+': [ # Group 15 plasma cell IgD- IgA+ IgG+ CD27+ CD38+ CD24(CR2)lo SLAMF7+ CD138(SDC1)+ HLA-DRA-- CD20(MS4A1)- Marked Plasma cells! Enrichr came back with lots of plasma cells https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6813733/
        'SDC1', # CD138 plasma
        'SLAMF7', # B cell/memory cell marker
        'FCRL5', # B cell activation
        'POU2AF1', # B cell specific transcription factor https://www.biolegend.com/de-at/products/purified-anti-pou2af1-obf1-antibody-12503
        'CD79A', # B cell receptor complex
        'CD38', # memory B cell marker
        'CD27', # plasma/memory B cell marker https://academic.oup.com/cei/article/213/2/164/6889379
    ],
    'GGB:PDAC_CL6': [ # Group 16 PDAC EMT?
        'MMP7', # poor prognosis in PDAC https://pubmed.ncbi.nlm.nih.gov/35343563/
        'CTGF', # elevated in PDAC, interacts with tumor stroma
        'VCAM1', # blockage inhibits PDAC https://pubmed.ncbi.nlm.nih.gov/33087490/
        'KRT23', # induced in differentiating PDAC https://pubmed.ncbi.nlm.nih.gov/11135429/
    ],
    'GGB:TUMOR_CL7': [ # Group 17 Ambiguous tumor
        'NOTUM', # negative regulator of Wnt signaling
        'APCDD1', # Wnt downstream target expression in colon cancer tissues
        'C6orf15', # PDAC predictive https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9006543/
        'NKDA1', # CAF associated https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10290535/
        'KRT6B', # squamous epithelial
    ],
    'GGB:TAM_CL1': [ # Group 18 macrophages CD14+ (macrophage/not neutrophil) CD33+ (neutrophil) CD86+ (M1/M2b macrophage) CD68+ (M1/TAM)
        'IL1A', # monocyte marker
        'IL1B', # monocyte marker
        'CD300E', # myeloid macrophage DC marker
        'CD83', # activated B/T, Treg, circulating DC, Langerhans
        'CD14', # DC monocytes macrophage
        'CD93', # endothelial neutrophils
        'CD86', # Monocytes, APCs, endothelial cells, activated B- and T-cells.	
        'CD33', # Normal myeloid cell progenitors, activated T-cells and NK cells.42
        'CD68', # Monocytes, macrophages, neutrophils, basophils, dendritic cells and myeloid progenitors. TAM specific marker
        'CD302', # Monocytes, macrophages, dendritic cells and granulocytes.	
        'CD163', # TAM
    ],
    'GGB:FIBROBLAST': [ # Group 19 mesenchymal fibroblast
        'PDGFRB', # mesenchymal marker (fibroblasts and mural cells)
        'DCN', # fibroblast marker
    ],
    'GGB:TAM_TREM2+_M1': [ # Group 20 TAM 
        'CD163', # TAM marker
        'TREM2', # TAM marker
        'C1QC', # TAM marker
        'TLR2', # M1 marker
        'SIGLEC1', # Macrophage marker
    ],
    'GGB:NK': [ # Group 21 T Enrichr NK call
        'SH2D1B', # NK marker
        'KLRF1', # NK marker
        'GNLY', # NK marker
        'CD247', # T/NK marker
        'GZMB', # cytotoxic T/NK
        'CD160', # cytotoxic T/NK
        'CD244', # T/NK cells
        'CD7', # T/NK cells 
    ],
    'GGB:Treg_FOXP3+CD4+': [ # Group 22 Treg FOXP3+CD4+CTLA4+TIGIT+ https://www.biorxiv.org/content/10.1101/2023.08.31.555730v1.full
        'FOXP3', # CD4+ Treg marker
        'CTLA4', # The one and only
        'TIGIT', # More immunotherapy hits
        'IZKF2', # Helios TF in FOXP3+CD4+
    ],
    'GGB:PDAC_CL8': [ # Group 23 PDAC (Cancer Cell Line Encylopedia / Azimuth)
        'UCA1', # tumor promoter
        'KLK6', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8392253/
    ],
    'GGB:PDAC_CL9': [ # Group 24 Seems cancerous but marked as T/NK? Maybe cycling? Lots of mitotic spindle program here?
        'HJURP', # https://www.nature.com/articles/s41419-020-2595-9
        'DLGAP5', # aggressive pancreatic cancer marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7414559/
        'BIRC5', # inhibits apoptosis https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7801710/
        'NUSAP1', # PDAC biomarker
        'AURKB', # https://pubmed.ncbi.nlm.nih.gov/21143115/
    ],
    'GGB:ENDOTHELIAL': [ # Group 25 endothelial / stem cell
        'CLEC14A', # cancer biomarker (CD144)
        'EMCN', # endothelial / stem cell
        'TEK', # oncogene https://link.springer.com/article/10.1186/1471-2105-5-81
        'CD34', # Hematopoietic stem cells and progenitors and capillary endothelial cells.	
        'ECSCR', # Endothelial specific https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3624410/
    ],
    'GGB:pDC': [ # Group 26 pDC
        'CLEC4C', # pDC marker https://pubmed.ncbi.nlm.nih.gov/30395816/ https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3195614/
        'LILRA4', # pDC marker
    ],
    'GGB:PDAC_CL10': [ # Group 27 PDAC Enrichr Cancer Cell Line Encyclopedia Wikipathway Pancreatic Cancer
        'KRT20', # https://www.neobiotechnologies.com/product/cytokeratin-20-krt20-colorectal-epithelial-marker-8/
        'TM4SF4', # beta cell marker?
        'DUOX2', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7943273/ https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5340089/ but reported as tumor suppresive https://www.nature.com/articles/s41598-022-07451-6
        'VNN1', # https://pubmed.ncbi.nlm.nih.gov/37114564/
        'CEACAM6', # PDAC marker https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8226832/
        'CEACAM5', # PDAC poor prognosis marker https://bmccancer.biomedcentral.com/articles/10.1186/s12885-022-10397-7
    ],
    'GGB:PDAC_CL11': [ # Group 28 PDAC Enrichr Cancer Cell Line Encyclopedia Wikipathway Pancreatic Cancer
        'GPT', # glutamine anabolism https://www.sciencedirect.com/science/article/pii/S2211124719312446
        'DPCR1', # upregulated or downregulated in tumor vs nontumor tissues, contradictory, https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6403508/ https://pubmed.ncbi.nlm.nih.gov/29242154/
        'CXCL5', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7065995/
        'TFF2', # enriched in PDAC https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7605628/ but classical marker
        'CEACAM5', # PDAC poor prognosis marker https://bmccancer.biomedcentral.com/articles/10.1186/s12885-022-10397-7
    ],
    'GGB:PDAC_CL12': [ # Group 29 PDAC Enrichr Wikipathway Pancreatic Cancer
        'PAEP', # expressed in NSCLC cancer https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6317686/
        'CALB2', # expressed in malignant mesothelioma https://bmccancer.biomedcentral.com/articles/10.1186/s12885-018-4385-7
        'SMC1B', # https://pubmed.ncbi.nlm.nih.gov/25216700/
    ],
    'GGB:cDC1_CLEC9A+_XCR1+': [ # Group 30 cDC1 CLEC9A+ XCR1+ THBD(CD141)+ 
        'CLEC9A', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2569177/
        'XCR1', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5674066/
        'THBD', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5674066/
    ],
    'GGB:PDAC_CL13': [ # Group 31 PDAC DUOX2+ CXCL5+
        'ZPLD1', # https://www.sciencedirect.com/science/article/pii/S2372770520300619    
        'DUOX2', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7943273/ https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5340089/ https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8825507/ but reported as tumor suppresive https://www.nature.com/articles/s41598-022-07451-6
        'CXCL5', # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7065995/
    ],
    'GGB:HEPATOCYTE': [ # Group 32 Ductal cell or hepatocyte?
        'CPB2', # hepatocyte marker
        'SLC17A4', # expressed in hepatocyte
        'ITIH2', # expressed in hepatocyte
    ],
    'GGB:PDAC_CL14': [ # Group 33 PDAC
        'S100A7', # upregulated in squamous cell carcinomas https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8265170/ https://www.spandidos-publications.com/ijo/50/5/1491
        'MUC15', # upregulated in cancer https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7570823/
        'KLK5', # less invasive PDAC phenotype https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3245846/
        'IL34', # high levels in PDAC patient plasma https://pubmed.ncbi.nlm.nih.gov/35316296/
    ],
    'GGB:TAM_CL2': [ # Group 34 Macrophage? CD163+ CD14+ CD68+
        'CD163', # https://www.abcam.com/primary-antibodies/human-cd-antigen-guide https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7017151/
        'CD14', # https://www.abcam.com/primary-antibodies/human-cd-antigen-guide
        'CD68', # TAMs
    ]
    
        
        
        
    
        
        
}

# VSIG4 TAM metabolite alteration locus

# perform KW test on cleared data

In [ ]:
coi = '30'
rgg_df = sc.get.rank_genes_groups_df(abl_trim, coi)
# print('CD' in sc.get.rank_genes_groups_df(abl_trim, coi).names.values)
print(sc.get.rank_genes_groups_df(abl_trim, coi).iloc[:30, :])
with pd.option_context('display.min_rows', 50):
    print(rgg_df.loc[[x.startswith('CCL2') for x in rgg_df.names.values], :])

print('\n'.join(sc.get.rank_genes_groups_df(abl_trim, coi).names[:100]))


In [ ]:
print(coarse_marker_genes)

# print(sc.tl.marker_gene_overlap(abl_trim, coarse_marker_genes, normalize='reference'))

GGB_cluster_dict = {idx:val for idx, val in enumerate(coarse_marker_genes)}
print(GGB_cluster_dict)

# I assigned these categories in cluster order 0-34

vector = [GGB_cluster_dict[int(x)] for x in abl_trim.obs.leiden]
# print(vector)
abl_trim.obs['GGB_cluster_dict'] = vector

In [ ]:
print(GGB_cluster_dict[17])

In [ ]:
sc.set_figure_params(figsize=(10, 8))
sc.pl.tsne(abl_trim, color=['leiden'], legend_loc='on data')
sc.pl.tsne(abl_trim, color=['Coarse_Cell_Annotations'], legend_loc='on data')
# sc.pl.tsne(abl_trim, color=['GGB_cluster_dict'])

In [ ]:
sc.pl.tsne(abl_trim, color=['leiden', 'donor_ID'])


In [ ]:
GGB_coarse_labels = {
    'GGB:B_CELL': ['GGB:B_CELL_NAIVE_ACTIVATED'],
    'GGB:DC': ['GGB:DC2_CD1C_CLEC10A_FCER1A'],
    'GGB:ENDOTHELIAL': ['GGB:ENDOTHELIAL'],
    'GGB:HEPATOCYTE': ['GGB:HEPATOCYTE'],
    'GGB:MACROPHAGE': ['GGB:MACROPHAGE_CD14',
                       'GGB:MACROPHAGE_TREM2+_M2',
                       'GGB:TAM_CL1', # This is over with the macrophages but has TAM aspects to it and single patient origin
                       'GGB:TAM_CL2', # This is over with the macrophages but has TAM aspects to it and single patient origin
                       'GGB:TAM_TREM2+_M1'],
    'GGB:MESENCHYMAL': ['GGB:FIBROBLAST'],
    'GGB:PLASMA_CELL': ['GGB:PC_PCreg_IgA+'],
    'GGB:T_NK': ['GGB:CD4_Tnaive_Treg', 
                 'GGB:CD4_Tnaive_Treg_2', 
                 'GGB:CD8_Tem_Teff',
                 'GGB:NK',
                 'GGB:NK_Teff'],
    'GGB:NET': ['GGB:NEUROENDOCRINE_TUMOR'],
    'GGB:T_REGS': ['GGB:Treg_FOXP3+CD4+'],
    'GGB:TUMOR': ['GGB:PDAC_CL1_BASAL',
                  'GGB:PDAC_CL2_BASAL',
                  'GGB:PDAC_CL3_TP63+_BASAL',
                  'GGB:PDAC_CL4',
                  'GGB:PDAC_CL5_CLASSICAL',
                  'GGB:PDAC_CL6',
                  'GGB:PDAC_CL7',
                  'GGB:PDAC_CL8',
                  'GGB:PDAC_CL9',
                  'GGB:PDAC_CL10',
                  'GGB:PDAC_CL11',
                  'GGB:PDAC_CL12',
                  'GGB:PDAC_CL13',
                  'GGB:PDAC_CL14',
                  'GGB:TUMOR_CL7',
                  'GGB:CAF'],  # original paper annotated this cluster with tumors. let's see how well it stacks up
    'GGB:XCR1_DC': ['GGB:cDC1_CLEC9A+_XCR1+'],
    'GGB:PDC_CELL': ['GGB:pDC'],
}
    

In [ ]:
GGB_coarse_inv = {}
for k, v in GGB_coarse_labels.items():
    for item in v:
        GGB_coarse_inv[item] = k
print(GGB_coarse_inv)

vector = [GGB_coarse_inv[k] for k in abl_trim.obs['GGB_cluster_dict']]

abl_trim.obs['GGB_coarse_cluster'] = vector

In [ ]:
print(abl_trim.obs['GGB_coarse_cluster'])

In [ ]:
print(abl_trim[abl_trim.obs['Coarse_Cell_Annotations'] == 'Tumor', :])

# print(abl_trim[abl_trim.obs['GGB_coarse_cluster'] == 'GGB:TUMOR', :])
# print(abl_trim[abl_trim.obs['GGB_coarse_cluster'] != 'GGB:TUMOR', :])

In [ ]:
abl_tumor = abl_trim[abl_trim.obs['Coarse_Cell_Annotations'] == 'Tumor', :].copy()
# PANFR0604 has no malignant cells, validated down below
abl_tumor = abl_tumor[abl_tumor.obs.donor_ID != 'PANFR0604', :]

sc.pp.highly_variable_genes(abl_tumor)
sc.pp.pca(abl_tumor)
sc.pp.neighbors(abl_tumor)
sc.tl.tsne(abl_tumor)
sc.tl.leiden(abl_tumor)

In [ ]:
sc.pl.tsne(abl_tumor, color=['leiden'], legend_loc="on data")

In [ ]:
print(Counter(abl_tumor.obs.leiden))

In [ ]:
sc.tl.rank_genes_groups(abl_tumor, 'leiden')

In [ ]:
coi = '5'
rgg_df = sc.get.rank_genes_groups_df(abl_tumor, coi)
# print('CD' in sc.get.rank_genes_groups_df(abl_trim, coi).names.values)
print(rgg_df.iloc[:30, :])
with pd.option_context('display.min_rows', 50):
    print(rgg_df.loc[[x.startswith('CD') for x in rgg_df.names.values], :])

# print('\n'.join(rgg_df.names[:100]))


# import itertools
# # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
# pdac_subtypes = {
#     'EXOCRINE_LIKE': ['REG1B', 'REG3A', 'REG1A', 'PNLIPRP2', 'CEL', 'PNLIP', 'PLA2G1B', 'CELA3A', 'CPB1', 'CELA3B', 'CTRB2', 'CLPS', 'CELA2B', 'PRSS2', 'PRSS1', 'GP2', 'SLC3A1', 'CFTR', 'SLC4A4', 'SPINK1'],
#     'CLASSICAL': ['AIM2', 'FAM26F', 'GPM6B', 'S100A2', 'KRT14', 'CAV1', 'LOX', 'SLC2A3', 'TWIST1', 'PAPPA', 'NT5E', 'CKS2', 'HMMR', 'SLC5A3', 'PMAIP1', 'PHLDA1', 'SLC16A1', 'FERMT1', 'HK2', 'AHNAK2'],
#     'QM_PDA': ['TMEM45B', 'SDR16C5', 'GPRC5A', 'AGR2', 'S100P', 'FXYD3', 'ST6GALNAC1', 'CEACAM5', 'CEACAM6', 'TFF1', 'TFF3', 'CAPN8', 'FOXQ1', 'ELF3', 'ERBB3', 'TSPAN8', 'TOX3', 'LGALS4', 'PLS1', 'GPX2', 'ATP10B', 'MUC13'],
# }

# all_markers = [x for x in list(itertools.chain(*list(pdac_subtypes.values()))) if x in abl_trim.var_names]

# for k in pdac_subtypes:
#     sc.pl.tsne(abl_tumor, color=[x for x in pdac_subtypes[k] if x in abl_tumor.var_names])


In [ ]:

import itertools
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
pdac_subtypes = {
    'EXOCRINE_LIKE': ['REG1B', 'REG3A', 'REG1A', 'PNLIPRP2', 'CEL', 'PNLIP', 'PLA2G1B', 'CELA3A', 'CPB1', 'CELA3B', 'CTRB2', 'CLPS', 'CELA2B', 'PRSS2', 'PRSS1', 'GP2', 'SLC3A1', 'CFTR', 'SLC4A4', 'SPINK1'],
    'CLASSICAL': ['AIM2', 'FAM26F', 'GPM6B', 'S100A2', 'KRT14', 'CAV1', 'LOX', 'SLC2A3', 'TWIST1', 'PAPPA', 'NT5E', 'CKS2', 'HMMR', 'SLC5A3', 'PMAIP1', 'PHLDA1', 'SLC16A1', 'FERMT1', 'HK2', 'AHNAK2'],
    'QM_PDA': ['TMEM45B', 'SDR16C5', 'GPRC5A', 'AGR2', 'S100P', 'FXYD3', 'ST6GALNAC1', 'CEACAM5', 'CEACAM6', 'TFF1', 'TFF3', 'CAPN8', 'FOXQ1', 'ELF3', 'ERBB3', 'TSPAN8', 'TOX3', 'LGALS4', 'PLS1', 'GPX2', 'ATP10B', 'MUC13'],
}

all_markers = [x for x in list(itertools.chain(*list(pdac_subtypes.values()))) if x in abl_trim.var_names]


for k in pdac_subtypes:
    sc.pl.tsne(abl_tumor, color=[x for x in pdac_subtypes[k] if x in abl_tumor.var_names])

In [ ]:
print(abl_tumor.obs.leiden)

In [ ]:

# import itertools
# # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
# pdac_subtypes = {
#     'EXOCRINE_LIKE': ['REG1B', 'REG3A', 'REG1A', 'PNLIPRP2', 'CEL', 'PNLIP', 'PLA2G1B', 'CELA3A', 'CPB1', 'CELA3B', 'CTRB2', 'CLPS', 'CELA2B', 'PRSS2', 'PRSS1', 'GP2', 'SLC3A1', 'CFTR', 'SLC4A4', 'SPINK1'],
#     'CLASSICAL': ['AIM2', 'FAM26F', 'GPM6B', 'S100A2', 'KRT14', 'CAV1', 'LOX', 'SLC2A3', 'TWIST1', 'PAPPA', 'NT5E', 'CKS2', 'HMMR', 'SLC5A3', 'PMAIP1', 'PHLDA1', 'SLC16A1', 'FERMT1', 'HK2', 'AHNAK2'],
#     'QM_PDA': ['TMEM45B', 'SDR16C5', 'GPRC5A', 'AGR2', 'S100P', 'FXYD3', 'ST6GALNAC1', 'CEACAM5', 'CEACAM6', 'TFF1', 'TFF3', 'CAPN8', 'FOXQ1', 'ELF3', 'ERBB3', 'TSPAN8', 'TOX3', 'LGALS4', 'PLS1', 'GPX2', 'ATP10B', 'MUC13'],
# }

# all_markers = [x for x in list(itertools.chain(*list(pdac_subtypes.values()))) if x in abl_trim.var_names]

moffitt_pdac = {
    'BASAL': [
        'VGLL1', 
        'UCA1', 
        'S100A2', 
        'LY6D', 
        'SPRR3',
        'SPRR1B',
        'LEMD1',
        'KRT15',
        'CTSV', # 'CTSL2',
        'DHRS9',
        'AREG',
        'CST6',
        'SERPINB3',
        # 'KRT6C', # not present in var_names
        'KRT6A',
        'FAM83A',
        'SCEL',
        'FGFBP1',
        'KRT7',
        'KRT17',
        'GPR87',
        'TNS4',
        'SLC2A1',
        'ANXA8L2',
    ],
    'CLASSICAL': [
        'BTNL8',
        'FAM3D',
        'PRR15L', # 'ATAD4',
        'AGR3',
        'CTSE',
        # 'TMEM238L', # 'LOC400573', not present in genes
        'LYZ',
        'TFF2',
        'TFF1',
        'ANXA10',
        'LGALS4',
        'PLA2G10',
        'CEACAM6',
        'VSIG2',
        'TSPAN8',
        'ST6GALNAC1',
        'AGR2',
        'TFF3',
        'CYP3A7',
        'MYO1A',
        'CLRN3',
        'KRT20',
        'CDH17',
        'SPINK4',
        'REG4',
    ],
}

# print(sc.tl.marker_gene_overlap(abl_tumor, pdac_subtypes, method='jaccard').T)



In [ ]:
verdict = sc.tl.marker_gene_overlap(abl_tumor, moffitt_pdac, method='overlap_coef').T
final_verdict = verdict > 0
final_verdict.loc[:, 'IC'] = np.logical_and(final_verdict.loc[:, 'BASAL'], final_verdict.loc[:, 'CLASSICAL'])
final_verdict.loc[:, 'SCORE'] = verdict.loc[:, 'CLASSICAL']-verdict.loc[:, 'BASAL']

buf = []
for idx in final_verdict.index:
    if final_verdict.loc[idx, 'IC']:
        buf.append('IC')
    elif final_verdict.loc[idx, 'CLASSICAL']:
        buf.append('CLASSICAL')
    elif final_verdict.loc[idx, 'BASAL']:
        buf.append('BASAL')        
    else:
        buf.append('NONE')
final_verdict.loc[:, 'VERDICT'] = buf
    

In [ ]:
print(final_verdict)

In [ ]:
abl_tumor.obs['moffitt_verdict'] = [final_verdict.loc[idx, 'VERDICT'] for idx in abl_tumor.obs.leiden]
abl_tumor.obs['moffitt_score'] = [final_verdict.loc[idx, 'SCORE'] for idx in abl_tumor.obs.leiden]    

In [ ]:
print(abl_tumor.obs.loc[:, ['donor_ID', 'moffitt_verdict']])
for grpid, grp in abl_tumor.obs.groupby('donor_ID'):
    print(grpid, Counter(grp.moffitt_verdict))

# One patient sample, PANFR0604, did not contain any malignant cells within the core biopsy used for scRNA-seq analysis.

In [ ]:
sc.set_figure_params(figsize=(10, 10))
sc.pl.tsne(abl_tumor, color=['moffitt_verdict', 'donor_ID'], legend_loc="on data")
sc.pl.tsne(abl_tumor, color=['moffitt_score', 'donor_ID'], legend_loc="on data")

In [ ]:
# print(abl_tumor)
abl_tumor.uns['leiden_rgg'] = abl_tumor.uns['rank_genes_groups']

In [ ]:
sc.tl.rank_genes_groups(abl_tumor, groupby='moffitt_verdict')

In [ ]:
sc.pl.rank_genes_groups_heatmap(abl_tumor, groupby='moffitt_verdict')

In [ ]:
# sc.tl.rank_genes_groups(abl_tumor, groupby='leiden')

In [ ]:
# sc.pl.rank_genes_groups_heatmap(abl_tumor, show_gene_labels=True)

In [ ]:
sc.pl.rank_genes_groups(abl_tumor, n_genes=15, sharey=False)
# IC group identifies unfavorable prognostic markers:
# KLK6, EZR, ASPH, S100P, 
# Also favorable markers MMP7, MTUS1
# IGF2BP1 is a testicular cancer marker

In [ ]:
sc.tl.score_genes(abl_tumor, moffitt_pdac['CLASSICAL'], score_name='classical_score', ctrl_size=100)
sc.tl.score_genes(abl_tumor, moffitt_pdac['BASAL'], score_name='basal_score', ctrl_size=100)
abl_tumor.obs.loc[:, 'diff_score'] = abl_tumor.obs.classical_score - abl_tumor.obs.basal_score

In [ ]:
print(abl_tumor.obs.diff_score)

In [ ]:
plt.plot(abl_tumor.obs.classical_score, abl_tumor.obs.basal_score, 'o')

In [ ]:
# abl_tumor.obs.classical_score.shape
# abl_tumor.to_df().loc[:, 'CEACAM6'].shape

import scipy.stats

classical_gene = 'CTSE'
basal_gene = 'KRT7'


plt.plot(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, classical_gene].values, '.')
plt.plot(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, classical_gene].values, '.')


plt.figure(1)
print(f'{classical_gene} (Classical)')
print('Pearson Classical', scipy.stats.pearsonr(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, classical_gene].values))
print('Pearson Basal', scipy.stats.pearsonr(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, classical_gene].values))

print('Spearman Classical', scipy.stats.spearmanr(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, classical_gene].values))
print('Spearman Basal', scipy.stats.spearmanr(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, classical_gene].values))

print()
plt.figure(2)
plt.plot(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, basal_gene].values, '.')
plt.plot(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, basal_gene].values, '.')

print(f'{basal_gene} (Basal)')
print('Pearson Classical', scipy.stats.pearsonr(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, basal_gene].values))
print('Pearson Basal', scipy.stats.pearsonr(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, basal_gene].values))

print('Spearman Classical', scipy.stats.spearmanr(abl_tumor.obs.classical_score, abl_tumor.to_df().loc[:, basal_gene].values))
print('Spearman Basal', scipy.stats.spearmanr(abl_tumor.obs.basal_score, abl_tumor.to_df().loc[:, basal_gene].values))

In [ ]:
# plt.scatter(abl_tumor.obs.classical_score, abl_tumor

In [ ]:
sc.set_figure_params(figsize=(10, 10))
sc.pl.tsne(abl_tumor, color='diff_score', legend_loc="on data")


In [ ]:
sc.pl.heatmap(abl_tumor, moffitt_pdac, 'diff_score', swap_axes=True, vmin=-4, vmax=4)

In [ ]:
sc.pl.heatmap(abl_tumor, moffitt_pdac, 'basal_score', swap_axes=True, vmin=-4, vmax=4)

In [ ]:
print([x for x in abl_tumor.obs.donor_ID.unique() if x not in ['PANFR0580', 'PANFR0588', 'PANFR0543']])

In [ ]:
sc.pl.pca(abl_tumor, color=['leiden', 'donor_ID'], size=50)


abl_no580 = abl_tumor[[x not in ['PANFR0580', 'PANFR0588', 'PANFR0543'] for x in abl_tumor.obs.donor_ID], :]
sc.pp.highly_variable_genes(abl_no580)
sc.pp.pca(abl_no580)
sc.pl.pca(abl_no580, color=['leiden', 'donor_ID'], size=50)


In [ ]:
sc.pp.neighbors(abl_no580, n_neighbors=35)
sc.tl.tsne(abl_no580)



In [ ]:
sc.tl.leiden(abl_no580, resolution=3.)

In [ ]:

sc.pl.tsne(abl_no580, color=['leiden', 'donor_ID'], size=100, legend_loc="on data")

In [ ]:
print(len(abl_no580.obs.donor_ID.unique()))
print(Counter(abl_no580.obs.donor_ID))

In [ ]:
with pd.option_context('display.min_rows', 50):
    print(pd.DataFrame(abl_no580.varm['PCs'][:, 0:3], index=abl_no580.var_names).sort_values(0))

In [ ]:
sc.pl.tsne(abl_no580, color='pct_counts_mt')


In [ ]:
abl_mttrim = abl_no580[abl_no580.obs.pct_counts_mt <= 20, :]

In [ ]:
sc.pp.highly_variable_genes(abl_mttrim)
sc.pp.pca(abl_mttrim)
sc.pp.neighbors(abl_mttrim)
sc.tl.tsne(abl_mttrim)




In [ ]:
sc.tl.leiden(abl_mttrim, resolution=2.)

In [ ]:
sc.pl.pca(abl_mttrim, color=['leiden', 'donor_ID'], size=100, legend_loc='on data')
sc.pl.tsne(abl_mttrim, color=['leiden', 'donor_ID'], size=100, legend_loc='on data')

In [ ]:
print(Counter(abl_mttrim.obs.donor_ID))

In [ ]:
sc.tl.umap(abl_mttrim)

In [ ]:
sc.pl.umap(abl_mttrim, color=['leiden', 'donor_ID'], size=100, legend_loc='on data')

In [ ]:
# PC0 is EMT :) :)
# PC1 is Classical :)
# PC2 is Basal :)


with pd.option_context('display.min_rows', 50):
    print(pd.DataFrame(abl_mttrim.varm['PCs'][:, 0:3], index=abl_mttrim.var_names).sort_values(2))

In [ ]:
print('\n'.join(pd.DataFrame(abl_mttrim.varm['PCs'], index=abl_mttrim.var_names).sort_values(2, ascending=False).index[:50]))

In [ ]:
sc.tl.score_genes(abl_mttrim, moffitt_pdac['CLASSICAL'], score_name='classical_score', ctrl_size=100)
sc.tl.score_genes(abl_mttrim, moffitt_pdac['BASAL'], score_name='basal_score', ctrl_size=100)
abl_mttrim.obs.loc[:, 'diff_score'] = abl_mttrim.obs.classical_score - abl_mttrim.obs.basal_score

In [ ]:
sc.pl.heatmap(abl_mttrim, moffitt_pdac, 'diff_score', swap_axes=True, vmin=-1.5, vmax=3)

In [ ]:
# pseudobulk = {}
# for grpid, grp in abl_tumor.obs.groupby('donor_ID'):
# #     print(grpid)
# #    print(pd.Series(abl_tumor[grp.index, :].layers['trimmed_counts'].sum(axis=0), abl_tumor.var_names))
#     pseudobulk[grpid] = pd.Series(abl_tumor[grp.index, :].layers['trimmed_counts'].sum(axis=0), abl_tumor.var_names)

In [ ]:
# pseudobulk = pd.DataFrame.from_dict(pseudobulk, orient='index')

In [ ]:
# pb_test = ad.AnnData(pseudobulk)
# pb_test.obs['donor_ID'] = pd.Categorical(pb_test.obs_names)

In [ ]:
# import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd

# Needed for some plotting
import matplotlib.pyplot as plt

# Plotting options, change to your liking
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))

In [ ]:
print(abl_trim.layers)

In [ ]:

# abl_tumor.layers['norm_counts'] = sc.pp.normalize

pdata = dc.pp.pseudobulk(
    abl_trim,
    sample_col='donor_ID',
    groups_col='GGB_coarse_cluster',
    layer='trimmed_counts',
    mode='sum',
    # min_cells=10,
    # min_counts=1000,
)

In [ ]:
print(pdata)

In [ ]:
print(pdata.obs.psbulk_n_cells)

In [ ]:
print(pdata.obs.psbulk_counts)

In [ ]:
dc.plot_psbulk_samples(pdata, groupby=['donor_ID', 'GGB_coarse_cluster'], figsize=(16, 8))


In [ ]:
print(pdata)

In [ ]:
sc.set_figure_params(figsize=(10, 8))

# Store raw counts in layers
pdata.layers['counts'] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
sc.pp.scale(pdata, max_value=10)
sc.tl.pca(pdata)

# Return raw counts to X
# dc.swap_layer(pdata, 'counts', X_layer_key=None, inplace=True)

In [ ]:
sc.pl.pca(pdata, color=['GGB_coarse_cluster'], ncols=1, size=300)
sc.pl.pca_variance_ratio(pdata)

In [ ]:
# pdata_bcell = pdata[pdata.obs.GGB_coarse_cluster == 'GGB:B_CELL', :]
# sc.pp.highly_variable_genes(pdata, flavor='seurat_v3', n_top_genes=2000)

In [ ]:
# sc.pp.pca(pdata, n_comps=10, use_highly_variable=True)

In [ ]:
sc.pl.pca(pdata, color=['donor_ID'], legend_loc="on data")

In [ ]:
sc.pp.neighbors(pdata)

In [ ]:
sc.tl.tsne(pdata, perplexity=30)


In [ ]:
sc.tl.leiden(pdata)

In [ ]:
sc.pl.tsne(pdata, color=['donor_ID', 'GGB_coarse_cluster'])

In [ ]:
print(pdata.X)

In [ ]:
pdata_clean = pdata[np.logical_and(pdata.obs.GGB_coarse_cluster != 'GGB:NET', pdata.obs.donor_ID != 'PANFR0526'), :].copy()
sc.tl.rank_genes_groups(pdata_clean, groupby='GGB_coarse_cluster')

In [ ]:
sc.tl.rank_genes_groups(pdata_clean, groupby='donor_ID')

In [ ]:
sc.pl.rank_genes_groups(pdata_clean, sharey=False)

In [ ]:
pdata_tumor = pdata[pdata.obs.GGB_coarse_cluster == 'GGB:TUMOR', :].copy()
print(pdata_tumor)
sc.pp.highly_variable_genes(pdata_tumor, flavor='seurat', n_top_genes=1000)

In [ ]:
sc.pp.pca(pdata_tumor, n_comps=20, use_highly_variable=True)


In [ ]:
sc.pl.pca(pdata_tumor, color=['donor_ID'], legend_loc="on data")

In [ ]:
sc.pp.neighbors(pdata_tumor)

In [ ]:
sc.tl.tsne(pdata_tumor, perplexity=20)

In [ ]:
sc.tl.leiden(pdata_tumor)

In [ ]:
sc.pl.tsne(pdata_tumor, color=['leiden'])

In [ ]:
samples = [pdata[pdata.obs.GGB_coarse_cluster == 'GGB:B_CELL', abl_trim.var.highly_variable] for cluster_id in abl_trim.obs.leiden.unique()]
others = [pdata[pdata.obs.leiden != cluster_id, abl_trim.var.highly_variable] for cluster_id in abl_trim.obs.leiden.unique()]

print(pdata.obs)

In [ ]:
# Import DESeq2
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

In [ ]:
coi = '27'
rgg_df = sc.get.rank_genes_groups_df(abl_tumor, coi)
# print('CD' in sc.get.rank_genes_groups_df(abl_trim, coi).names.values)
print(sc.get.rank_genes_groups_df(abl_tumor, coi).iloc[:30, :])
with pd.option_context('display.min_rows', 50):
    print(rgg_df.loc[[x.startswith('CD') for x in rgg_df.names.values], :])

print('\n'.join(sc.get.rank_genes_groups_df(abl_tumor, coi).names[:100]))


In [ ]:
sc.get.rank_genes_groups_df(abl_trim, coi).set_index('names').loc['FOXP3', :]